# B06 · Plotly 交互式图表

> 阶段〇第 6 周。matplotlib 出的是「印进报告的静态图」；
> Plotly 出的是「能缩放、能悬浮读数、能转 3D、能播放动画」的交互图——
> 调参、看长曲线细节、给别人演示仿真结果时，交互图效率高得多。

## 学习目标

1. 区分 Plotly 的两层 API：`plotly.express`（快）与 `graph_objects`（灵活）；
2. 与 pandas 联动画交互折线/散点；
3. 用 `make_subplots` 组合多子图；
4. 画 3D 曲面图与「根轨迹风格」的 s 平面极点图；
5. 做带滑块/播放键的简单动画；
6. 在 Jupyter 中渲染，并把图导出为独立 HTML 文件分享。

> 说明：Plotly 在 Jupyter 里用默认渲染器即可（输出内嵌在 notebook 中，无需联网加载 plotly.js——
> 本环境 plotly ≥ 6 已自带）。交互功能（缩放、悬浮）需要在 Jupyter 里运行才能体验，
> 静态阅读时看到的是首帧画面。

## 1. 两层 API 与 pandas 联动

- `plotly.express`（惯例 `import plotly.express as px`）：一行出图，**吃「长表」DataFrame**
  （每行一个观测，列是变量）；
- `plotly.graph_objects`（惯例 `import plotly.graph_objects as go`）：逐条 trace 精细控制。

先造数据：三个时间常数的一阶环节阶跃响应，组织成长表。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
RUNS = PROJECT_ROOT / "runs" / "00_basic"
RUNS.mkdir(parents=True, exist_ok=True)

t = np.linspace(0, 5, 501)
rows = []
for tau in [0.2, 0.5, 1.0]:
    y = 2 * (1 - np.exp(-t / tau))
    rows.append(pd.DataFrame({"t": t, "y": y, "tau": f"tau={tau}"}))
df = pd.concat(rows, ignore_index=True)   # 长表：t / y / tau 三列
df.head()

,t,y,tau
0,0.00,0.000000,tau=0.2
1,0.01,0.097541,tau=0.2
2,0.02,0.190325,tau=0.2
3,0.03,0.278584,tau=0.2
4,0.04,0.362538,tau=0.2


In [2]:
# plotly.express：一行出图，按 tau 列自动分组配色
fig = px.line(df, x="t", y="y", color="tau",
              title="First-order step response (hover to read values)",
              labels={"t": "Time (s)", "y": "Output y"})
fig.show()
# 试试：悬浮读数 / 框选放大 / 双击图例单独显示某条曲线

## 2. `graph_objects`：逐条 trace 精细控制

需要自定义每个点的样式、加参考线、叠多种图形时用 `go.Figure`。

In [3]:
t_c = np.linspace(0, 5, 51)                       # 粗采样点
y_c = 2 * (1 - np.exp(-t_c / 0.5))
y_fine = 2 * (1 - np.exp(-t / 0.5))

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=y_fine, mode="lines", name="continuous",
                         line=dict(width=2)))
fig.add_trace(go.Scatter(x=t_c, y=y_c, mode="markers", name="samples",
                         marker=dict(size=7, symbol="x")))
# 加稳态线与 2% 误差带
fig.add_hline(y=2.0, line_dash="dash", annotation_text="steady state")
fig.add_hrect(y0=1.96, y1=2.04, fillcolor="green", opacity=0.1, line_width=0)
fig.update_layout(title="Step response with 2% band",
                  xaxis_title="Time (s)", yaxis_title="y", template="plotly_white")
fig.show()

## 3. 子图：`make_subplots`

监控系统经典布局：上图输出、下图控制量，共享 x 轴。

In [4]:
from plotly.subplots import make_subplots

# 用一阶环节 + 简单 P 控制的仿真数据（不用真解，造数即可）
K, tau, Kp, r = 2.0, 0.5, 3.0, 1.0
dt = 0.005
n = 2001
tt = np.linspace(0, 10, n)
y = np.zeros(n); u = np.zeros(n)
for k in range(1, n):
    e = r - y[k - 1]
    u[k] = Kp * e
    y[k] = y[k - 1] + dt * (K * u[k] - y[k - 1]) / tau

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Output y", "Control u"))
fig.add_trace(go.Scatter(x=tt, y=y, name="y"), row=1, col=1)
fig.add_trace(go.Scatter(x=tt, y=r * np.ones_like(tt), name="setpoint",
                         line=dict(dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=tt, y=u, name="u", line=dict(color="orange")), row=2, col=1)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_layout(height=500, title="P control: output & effort", template="plotly_white")
fig.show()
print("稳态误差（P 控制必有静差）:", round(r - y[-1], 4))

稳态误差（P 控制必有静差）: 0.1429


## 4. 3D 曲面图

二元函数 $z = \sin(x)\cos(y)$——在 RL 里这就是价值函数/损失曲面的迷你版；
旋转它，直观理解「曲面」这个概念（以后看 loss landscape 论文图就不慌了）。

In [5]:
x = np.linspace(-3, 3, 80)
yv = np.linspace(-3, 3, 80)
Xg, Yg = np.meshgrid(x, yv)
Z = np.sin(Xg) * np.cos(Yg)

fig = go.Figure(go.Surface(x=x, y=yv, z=Z, colorscale="Viridis"))
fig.update_layout(title="z = sin(x) cos(y)", height=500,
                  scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"))
fig.show()
# 拖拽旋转、滚轮缩放

## 5. 「根轨迹风格」图：s 平面上的极点移动

经典控制：单位负反馈下开环 $G(s) = \dfrac{K}{s(s+2)}$，
闭环特征方程 $s^2 + 2s + K = 0$，极点 $s = -1 \pm \sqrt{1 - K}$。
$K$ 从 0 增大：两极点沿实轴靠拢 → 汇合于 $-1$（临界）→ 分裂为共轭复根垂直上升（欠阻尼）。
用散点颜色编码 $K$，就是一张迷你根轨迹。

In [6]:
Ks = np.concatenate([np.linspace(0.1, 1.0, 40), np.linspace(1.05, 10, 60)])
poles_re, poles_im, poles_K = [], [], []
for Kv in Ks:
    roots = np.roots([1, 2, Kv])          # s^2 + 2s + K = 0
    for rt in roots:
        poles_re.append(rt.real); poles_im.append(rt.imag); poles_K.append(Kv)

fig = go.Figure(go.Scatter(
    x=poles_re, y=poles_im, mode="markers",
    marker=dict(size=5, color=poles_K, colorscale="Plasma", colorbar=dict(title="K")),
    name="closed-loop poles"))
fig.add_vline(x=0, line_dash="dash", line_color="red",
              annotation_text="imaginary axis (stability boundary)")
fig.update_layout(title="Root locus style: poles of s^2 + 2s + K = 0",
                  xaxis_title="Real", yaxis_title="Imaginary", template="plotly_white")
fig.show()
print("所有极点实部 < 0 → K>0 时闭环恒稳定（本例）；虚部非零 → 欠阻尼振荡")

所有极点实部 < 0 → K>0 时闭环恒稳定（本例）；虚部非零 → 欠阻尼振荡


## 6. 简单动画：滑块播放相位变化

动画 = 一组 `Frame` + `sliders`（拖动）/ `updatemenus`（播放键）。
例：正弦波 $y = \sin(x + \varphi)$，相位 $\varphi$ 从 0 播到 $\pi$。

In [7]:
x = np.linspace(0, 2 * np.pi, 200)
phases = np.linspace(0, np.pi, 25)

frames = [go.Frame(data=[go.Scatter(x=x, y=np.sin(x + phi))],
                   name=f"{phi:.2f}") for phi in phases]

fig = go.Figure(
    data=[go.Scatter(x=x, y=np.sin(x))],
    frames=frames)
fig.update_layout(
    title="Sine wave: phase animation",
    yaxis=dict(range=[-1.3, 1.3]), template="plotly_white",
    updatemenus=[dict(type="buttons", showactive=False, x=0.05, y=1.15,
                      buttons=[dict(label="Play", method="animate",
                                    args=[None, {"frame": {"duration": 80},
                                                 "fromcurrent": True}]),
                               dict(label="Pause", method="animate",
                                    args=[[None], {"mode": "immediate"}])])],
    sliders=[dict(active=0, x=0.1, len=0.85,
                  steps=[dict(method="animate",
                              args=[[f"{p:.2f}"],
                                    {"mode": "immediate", "frame": {"duration": 0}}],
                              label=f"{p:.2f}") for p in phases])])
fig.show()
# 点 Play 播放，或拖动底部滑块；帧 name 与 slider step 的 label 一一对应

## 7. 导出 HTML：把交互图发给别人

`fig.write_html()` 生成**自包含**网页——双击就能在浏览器里交互，
适合附在实验报告、邮件、issue 里。产物照旧放 `runs/`。

In [8]:
html_path = RUNS / "b06_root_locus.html"
fig_rl = go.Figure(go.Scatter(x=poles_re, y=poles_im, mode="markers",
                              marker=dict(size=5, color=poles_K, colorscale="Plasma")))
fig_rl.update_layout(title="Root locus (exported HTML)",
                     xaxis_title="Real", yaxis_title="Imaginary")
fig_rl.write_html(html_path, include_plotlyjs="cdn")   # CDN 版体积小；离线分享用 include_plotlyjs=True
print("已导出:", html_path)
print("浏览器打开即可交互（缩放/悬浮读数）")

已导出: /data/wangf/robot_rl_learn/runs/00_basic/b06_root_locus.html
浏览器打开即可交互（缩放/悬浮读数）


## 小结与衔接

- 快用 `px`、精调 `go`；长表 DataFrame 是 `px` 的最佳拍档；
- 交互图三件套：悬浮读数、框选缩放、图例开关——调参看细节离不开；
- 3D 曲面与动画滑块是展示「参数如何影响结果」的利器；
- `write_html` 让交互图脱离 notebook 独立分享。

**下周 B07**：把仿真整个搬进浏览器——Streamlit 做一个滑块调 Kp/Ki/Kd 的 PID 面板。

---

## ✏️ 练习

> 规则：先独立完成，再点开折叠的参考答案核对。

**练习 1（★，10 分钟，10 分）——长表 + px**
生成三个增益 $K \in \{0.5, 1, 2\}$（$\tau=0.5$）的一阶阶跃响应长表，
用 `px.line` 按 K 分组出图，标题和轴标签齐全。
**交付物**：交互图。

**练习 2（★★，20 分钟，20 分）——超调量随阻尼比变化**
二阶系统 $\omega_n = 2$，对 $\zeta \in \{0.1, 0.2, ..., 0.9\}$ 用 `solve_ivp` 仿真阶跃响应并算超调量 $M_p$，
用 `go.Figure` 画 $M_p$ 随 $\zeta$ 的散点+连线图（散点悬浮显示具体数值）。
**交付物**：交互图 + $\zeta=0.3$ 时的 $M_p$ 读数（理论约 37%）。

**练习 3（★，15 分钟，10 分）——3D 看损失曲面**
画 $J(\theta_0, \theta_1) = (\theta_0 - 1)^2 + 2(\theta_1 + 1)^2$ 的 3D 曲面（$\theta \in [-3, 3]$），
找到最低点坐标并口述：这对应梯度下降在找什么？
**交付物**：曲面图 + 最低点坐标 $(1, -1)$。

**练习 4（★★，20 分钟，15 分）——单摆动画**
用 B05 的非线性单摆仿真结果（初摆角 60°），做一个动画：
每帧画摆杆（从原点到 $(L\sin\theta, -L\cos\theta)$ 的线段），滑块拖动看单摆来回摆动。
**交付物**：可播放/可拖动的动画。

---

<details>
<summary>参考答案（做完再点开）</summary>

**练习 1**：

```python
rows = [pd.DataFrame({"t": t, "y": K * (1 - np.exp(-t / 0.5)), "K": f"K={K}"})
        for K in [0.5, 1, 2]]
px.line(pd.concat(rows), x="t", y="y", color="K",
        labels={"t": "Time (s)"}, title="Step response vs K").show()
```

**练习 2**：

```python
from scipy.integrate import solve_ivp
zeta_list = np.arange(0.1, 1.0, 0.1)
Mp_list = []
for z in zeta_list:
    wn = 2.0
    f = lambda t, s: [s[1], wn**2 * (1 - s[0]) - 2 * z * wn * s[1]]
    s = solve_ivp(f, (0, 10), [0, 0], t_eval=np.linspace(0, 10, 2000))
    Mp_list.append((s.y[0].max() - 1) * 100)
go.Figure(go.Scatter(x=zeta_list, y=Mp_list, mode="markers+lines")).show()
# ζ=0.3 时 Mp ≈ 37.2%，与理论 exp(-πζ/√(1-ζ²)) 一致
```

**练习 3**：

```python
th0 = np.linspace(-3, 3, 60); th1 = np.linspace(-3, 3, 60)
T0, T1 = np.meshgrid(th0, th1)
J = (T0 - 1) ** 2 + 2 * (T1 + 1) ** 2
go.Figure(go.Surface(x=th0, y=th1, z=J)).show()
# 最低点 (1, -1)；梯度下降就是沿曲面最陡方向往下走到这个碗底
```

**练习 4**：

```python
sol = solve_ivp(pendulum_nl, (0, 6), [np.deg2rad(60), 0], t_eval=np.linspace(0, 6, 121))
th = sol.y[0]
def rod(theta):
    return go.Scatter(x=[0, np.sin(theta)], y=[0, -np.cos(theta)], mode="lines+markers")
frames = [go.Frame(data=[rod(a)], name=str(i)) for i, a in enumerate(th)]
fig = go.Figure(data=[rod(th[0])], frames=frames)
fig.update_layout(xaxis=dict(range=[-1.2, 1.2]), yaxis=dict(range=[-1.2, 0.3], scaleanchor="x"),
                  sliders=[dict(steps=[dict(method="animate",
                                            args=[[str(i)], {"mode": "immediate"}],
                                            label=str(i)) for i in range(len(th))])])
fig.show()
```

</details>

---

## 延伸阅读

- [Plotly Python 官方文档](https://plotly.com/python/)
- [plotly.express 速查](https://plotly.com/python/plotly-express/)
- [Plotly 动画教程](https://plotly.com/python/animations/)
- [Plotly 3D 曲面](https://plotly.com/python/3d-surface-plots/)